In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
master = pd.read_csv(r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Data\master_tourism_dataset.csv")



In [2]:
master.head()

,TransactionId,UserId,VisitYear,VisitMonth,VisitModeId,AttractionId,Rating,ContinentId,RegionId,CountryId,...,AttractionAddress,AttractionType,CityName,Country,Region,Continent,TotalVisits,UserAvgRating,AttractionPopularity,AttractionAvgRating
0,3,70456,2022,10,2,640,5,5,21,163,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Guildford,United Kingdom,Western Europe,Europe,1,5.0,13197,4.26703
1,8,7567,2022,10,4,640,5,2,8,48,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Ontario,Canada,Northern America,America,1,5.0,13197,4.26703
2,9,79069,2022,10,3,640,5,2,9,54,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Brazil,Brazil,South America,America,1,5.0,13197,4.26703
3,10,31019,2022,10,3,640,3,5,17,135,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Zurich,Switzerland,Central Europe,Europe,2,3.0,13197,4.26703
4,15,43611,2022,10,2,640,3,5,21,163,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Manchester,United Kingdom,Western Europe,Europe,3,3.0,13197,4.26703


In [3]:
# Columns to drop for regression model
drop_cols = [
    'TransactionId',
    'UserId',
    'AttractionId',
    'Attraction',
    'AttractionAddress'
]

master_ml = master.drop(columns=drop_cols)


In [4]:
print(master_ml.columns)
print(master_ml.shape)


Index(['VisitYear', 'VisitMonth', 'VisitModeId', 'Rating', 'ContinentId',
       'RegionId', 'CountryId', 'CityId', 'VisitMode', 'AttractionCityId',
       'AttractionTypeId', 'AttractionType', 'CityName', 'Country', 'Region',
       'Continent', 'TotalVisits', 'UserAvgRating', 'AttractionPopularity',
       'AttractionAvgRating'],
      dtype='object')
(52922, 20)


In [5]:
feature_cols = [
    'VisitYear',
    'VisitMonth',
    'VisitMode',
    'Continent',
    'Region',
    'Country',
    'CityName',
    'AttractionType',
    'TotalVisits',
    'UserAvgRating',
    'AttractionPopularity',
    'AttractionAvgRating'
]

X = master_ml[feature_cols]
y = master_ml['Rating']


In [6]:
print(master_ml.columns)
print(master_ml.shape)


Index(['VisitYear', 'VisitMonth', 'VisitModeId', 'Rating', 'ContinentId',
       'RegionId', 'CountryId', 'CityId', 'VisitMode', 'AttractionCityId',
       'AttractionTypeId', 'AttractionType', 'CityName', 'Country', 'Region',
       'Continent', 'TotalVisits', 'UserAvgRating', 'AttractionPopularity',
       'AttractionAvgRating'],
      dtype='object')
(52922, 20)


In [7]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    'VisitMode',
    'Continent',
    'Region',
    'Country',
    'CityName',
    'AttractionType'
]

master_encoded = master_ml[feature_cols].copy()

le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    master_encoded[col] = le.fit_transform(master_encoded[col])
    le_dict[col] = le


In [8]:
X = master_encoded
y = master_ml[['Rating']]



In [9]:
X.head()

,VisitYear,VisitMonth,VisitMode,Continent,Region,Country,CityName,AttractionType,TotalVisits,UserAvgRating,AttractionPopularity,AttractionAvgRating
0,2022,10,1,4,21,144,1801,8,1,5.0,13197,4.26703
1,2022,10,3,1,12,27,3509,8,1,5.0,13197,4.26703
2,2022,10,2,1,15,21,630,8,1,5.0,13197,4.26703
3,2022,10,2,4,6,132,5463,8,2,3.0,13197,4.26703
4,2022,10,1,4,21,144,2847,8,3,3.0,13197,4.26703


In [10]:
y.head()

,Rating
0,5
1,5
2,5
3,3
4,3


In [11]:
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Second split: Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)



Train shape: (37045, 12)
Validation shape: (7938, 12)
Test shape: (7939, 12)


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

# Validation prediction
y_val_pred_lr = lr.predict(X_val)

mse_lr = mean_squared_error(y_val, y_val_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_val, y_val_pred_lr)

print("Linear Regression Performance (Validation)")
print("MSE:", mse_lr)
print("RMSE:", rmse_lr)
print("R2:", r2_lr)


Linear Regression Performance (Validation)
MSE: 0.2581102670912745
RMSE: 0.5080455364347516
R2: 0.7288779822893264


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

y_val_pred_rf = rf.predict(X_val)

mse_rf = mean_squared_error(y_val, y_val_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_val, y_val_pred_rf)

print("\nRandom Forest Performance (Validation)")
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Random Forest Performance (Validation)
MSE: 0.2940238324574767
RMSE: 0.5422396448596106
R2: 0.6911539567594716


In [14]:
from xgboost import XGBRegressor

xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)

y_val_pred_xgb = xgb.predict(X_val)

mse_xgb = mean_squared_error(y_val, y_val_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_val, y_val_pred_xgb)

print("XGBoost Performance (Validation)")
print("MSE:", mse_xgb)
print("RMSE:", rmse_xgb)
print("R2:", r2_xgb)


XGBoost Performance (Validation)
MSE: 0.2747921645641327
RMSE: 0.5242062233168667
R2: 0.7113550901412964


In [33]:
import joblib

joblib.dump(rf, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\regression_model.pkl")


['C:\\Users\\kisho\\Downloads\\Tourism_Experience_Analytics_Project\\Models\\regression_model.pkl']

## Classification model

In [16]:
master_clf = master.copy()

drop_cols = [
    'TransactionId',
    'UserId',
    'AttractionId',
    'Attraction',
    'AttractionAddress',
    'Rating',
    'UserAvgRating',
    'AttractionAvgRating',
    'VisitModeId'
]

master_clf = master_clf.drop(columns=drop_cols)


In [17]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    'Continent',
    'Region',
    'Country',
    'CityName',
    'AttractionType',
    'VisitMode'   # This is target but needs encoding
]

master_clf_encoded = master_clf.copy()

le_dict_clf = {}

for col in categorical_cols:
    le = LabelEncoder()
    master_clf_encoded[col] = le.fit_transform(master_clf_encoded[col])
    le_dict_clf[col] = le


In [18]:
X = master_clf_encoded.drop('VisitMode', axis=1)
y = master_clf_encoded['VisitMode']


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Train: (37045, 15)
Val: (7938, 15)
Test: (7939, 15)


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_clf = LogisticRegression(max_iter=1000)
log_clf.fit(X_train, y_train)

y_val_pred_log = log_clf.predict(X_val)

print("Logistic Regression Accuracy:",
      accuracy_score(y_val, y_val_pred_log))

print(classification_report(y_val, y_val_pred_log))


Logistic Regression Accuracy: 0.43008314436885864
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        94
           1       0.43      0.87      0.57      3242
           2       0.43      0.27      0.33      2282
           3       0.22      0.00      0.00      1641
           4       0.00      0.00      0.00       679

    accuracy                           0.43      7938
   macro avg       0.22      0.23      0.18      7938
weighted avg       0.35      0.43      0.33      7938



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wi

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)

y_val_pred_rf = rf_clf.predict(X_val)

print("Random Forest Accuracy:",
      accuracy_score(y_val, y_val_pred_rf))

print(classification_report(y_val, y_val_pred_rf))


Random Forest Accuracy: 0.5036533131771227
              precision    recall  f1-score   support

           0       0.45      0.21      0.29        94
           1       0.53      0.68      0.60      3242
           2       0.53      0.48      0.50      2282
           3       0.40      0.32      0.36      1641
           4       0.41      0.23      0.29       679

    accuracy                           0.50      7938
   macro avg       0.47      0.38      0.41      7938
weighted avg       0.49      0.50      0.49      7938



In [22]:
master['VisitMode'].value_counts()

VisitMode
Couples     21617
Family      15215
Friends     10944
Solo         4523
Business      623
Name: count, dtype: int64

In [23]:
rf_clf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

rf_clf.fit(X_train, y_train)

y_val_pred_rf = rf_clf.predict(X_val)

from sklearn.metrics import accuracy_score, classification_report

print("Balanced RF Accuracy:",
      accuracy_score(y_val, y_val_pred_rf))

print(classification_report(y_val, y_val_pred_rf))


Balanced RF Accuracy: 0.4945830183925422
              precision    recall  f1-score   support

           0       0.38      0.26      0.31        94
           1       0.53      0.66      0.59      3242
           2       0.52      0.47      0.49      2282
           3       0.39      0.32      0.35      1641
           4       0.36      0.24      0.28       679

    accuracy                           0.49      7938
   macro avg       0.44      0.39      0.41      7938
weighted avg       0.48      0.49      0.48      7938



In [24]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_clf.fit(X_train, y_train)

y_val_pred_xgb = xgb_clf.predict(X_val)

print("XGBoost Accuracy:",
      accuracy_score(y_val, y_val_pred_xgb))

print(classification_report(y_val, y_val_pred_xgb))


C:\ProgramData\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.5170068027210885
              precision    recall  f1-score   support

           0       0.60      0.10      0.17        94
           1       0.51      0.80      0.63      3242
           2       0.56      0.47      0.51      2282
           3       0.44      0.22      0.30      1641
           4       0.56      0.09      0.16       679

    accuracy                           0.52      7938
   macro avg       0.54      0.34      0.35      7938
weighted avg       0.52      0.52      0.48      7938



In [34]:
joblib.dump(rf_clf, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\classification_model.pkl")


['C:\\Users\\kisho\\Downloads\\Tourism_Experience_Analytics_Project\\Models\\classification_model.pkl']

In [27]:
model_test = joblib.load(r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\regression_model.pkl")
print(model_test)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)


In [29]:
joblib.dump(le_dict, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\encoders.pkl")

['C:\\Users\\kisho\\Downloads\\Tourism_Experience_Analytics_Project\\Models\\encoders.pkl']

In [30]:
joblib.dump(le_dict_clf, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\encoders_classification.pkl")

['C:\\Users\\kisho\\Downloads\\Tourism_Experience_Analytics_Project\\Models\\encoders_classification.pkl']

In [32]:
print(xgb.get_booster().feature_names)

['VisitYear', 'VisitMonth', 'VisitMode', 'Continent', 'Region', 'Country', 'CityName', 'AttractionType', 'TotalVisits', 'UserAvgRating', 'AttractionPopularity', 'AttractionAvgRating']


In [36]:
feature_cols = rf.feature_names_in_
print(feature_cols)

['VisitYear' 'VisitMonth' 'VisitMode' 'Continent' 'Region' 'Country'
 'CityName' 'AttractionType' 'TotalVisits' 'UserAvgRating'
 'AttractionPopularity' 'AttractionAvgRating']


In [37]:
X_clf = master[feature_cols].copy()
y_clf = master['VisitMode']   # encoded version

In [39]:
import joblib

encoders = joblib.load(r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\encoders.pkl")

In [40]:
y_clf = encoders['VisitMode'].transform(master['VisitMode'])

In [49]:
master_clf_encoded = X_clf.copy()
le_dict_clf = {}

for col in feature_cols:
    le = LabelEncoder()
    master_clf_encoded[col] = le.fit_transform(master_clf_encoded[col])
    le_dict_clf[col] = le


In [52]:
X_clf =master_clf_encoded 

In [53]:
from sklearn.ensemble import RandomForestClassifier

rf_clf_new = RandomForestClassifier(random_state=42)

rf_clf_new.fit(X_clf.drop(columns=['VisitMode']), y_clf)

RandomForestClassifier(random_state=42)

In [55]:
joblib.dump(rf_clf_new, r"C:\Users\kisho\Downloads\Tourism_Experience_Analytics_Project\Models\classification_model.pkl")

['C:\\Users\\kisho\\Downloads\\Tourism_Experience_Analytics_Project\\Models\\classification_model.pkl']

In [51]:
X_clf.head()

,VisitYear,VisitMonth,VisitMode,Continent,Region,Country,CityName,AttractionType,TotalVisits,UserAvgRating,AttractionPopularity,AttractionAvgRating
0,2022,10,Couples,Europe,Western Europe,United Kingdom,Guildford,Nature & Wildlife Areas,1,5.0,13197,4.26703
1,2022,10,Friends,America,Northern America,Canada,Ontario,Nature & Wildlife Areas,1,5.0,13197,4.26703
2,2022,10,Family,America,South America,Brazil,Brazil,Nature & Wildlife Areas,1,5.0,13197,4.26703
3,2022,10,Family,Europe,Central Europe,Switzerland,Zurich,Nature & Wildlife Areas,2,3.0,13197,4.26703
4,2022,10,Couples,Europe,Western Europe,United Kingdom,Manchester,Nature & Wildlife Areas,3,3.0,13197,4.26703
